# 🚨 Multimodal Crisis Intelligence & Emergency Triage
### End-to-End Deep Learning: Fusing Social Text (`BERTweet`) and Photos (`ConvNeXt-Tiny`) via a Gated Multimodal Unit (GMU)

This notebook demonstrates:
1. **Environment Setup:** Installing PyTorch, Hugging Face Transformers, and Timm on GPU.
2. **Dataset Loading:** Paired disaster texts and images with Event-Held-Out splits.
3. **Model Architecture:** Gated Multimodal Unit (GMU) with multi-task prediction heads.
4. **Training & Multi-Task Loss:** Cross-Entropy (Scenario) + BCEWithLogits (Resource Needs).
5. **Held-Out Evaluation:** Computing Macro-F1, Accuracy, and Gated Modality Attribution.
6. **Interactive Inference:** Testing text + image queries with confidence calibration.

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers timm scikit-learn pillow

In [ ]:
# Step 2: Check GPU Acceleration
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Run Data Preparation
!python data/download_and_prep.py

In [ ]:
# Step 4: Train the Multimodal Fusion Model
# Note: Running on CUDA with BERTweet and ConvNeXt-Tiny backbones
!python train.py --epochs 5 --batch-size 8 --lr 2e-5 --device cuda

In [ ]:
# Step 5: Run Interactive Multimodal Inference
from inference import CrisisPredictor
from PIL import Image

predictor = CrisisPredictor()

# Test Case: Flooded Underpass
test_text = "Akhbarnagar underpass completely submerged in 8 feet water. Car floating with 2 passengers trapped on roof!"
test_img = Image.open("data/images/t2_urban_flooding.jpg")

result = predictor.predict(text=test_text, image=test_img)

print("="*50)
print(f"Predicted Scenario: {result['predicted_scenario']} (Confidence: {result['confidence']*100:.1f}%)")
print(f"Gated Attribution: {result['gated_attribution']}")
print("Urgent Needs Detected:")
for need, info in result['resource_needs'].items():
    if info['detected']:
        print(f"  [!] {need.replace('_', ' ').title()} ({info['probability']*100:.1f}%)")
print("="*50)